> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 3 · Notebook 07 — Time series with pandas and Polars

**Sessions:** S18 (pandas & Polars for time series) · [Lesson plan](../../docs/lessons/PART_03_PYTHON_ENGINEERING.md) · graded labs in [`labs/part03/`](../../labs/part03/)

**You will:**
1. Resample trades into OHLCV bars.
2. Join each trade to the quote in force at that moment with `merge_asof`, and see how a careless join leaks the future.
3. Measure effective spreads.
4. Compute VWAP bars with Polars and compare speed.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p3lib.py is in notebooks/part03/
    sys.path.insert(0, str(d))
from decimal import Decimal
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p3lib as p

p.use_course_style()

In [ ]:
trades, quotes = p.trades_and_quotes()
print(f"{len(trades)} trades, {len(quotes)} quotes, UTC timestamps")
trades.head(3)

## 1. OHLCV bars

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
g = trades.set_index("time").resample("5min")
bars = g["price"].ohlc()
bars["volume"] = g["size"].sum()
bars = bars.dropna()
bars = p.check("5-minute OHLCV bars", bars, p.ohlcv(trades))
bars.head()

## 2. As-of joins: the quote in force at each trade

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
tq = pd.merge_asof(trades.sort_values("time"), quotes.sort_values("time"), on="time", direction="backward")
tq = p.check("merge_asof (no look-ahead)", tq, p.with_quotes_asof(trades, quotes))
tq.head()

In [ ]:
q2 = quotes.rename(columns={"time": "quote_time"})
nearest = pd.merge_asof(trades, q2, left_on="time", right_on="quote_time", direction="nearest")
leak = (nearest["quote_time"] > nearest["time"]).mean()
print(f"direction='nearest' used a quote from the FUTURE for {leak:.0%} of trades — a silent look-ahead.")
tq["mid"] = (tq["bid"] + tq["ask"]) / 2
tq["eff_spread_bps"] = 2 * (tq["price"] - tq["mid"]).abs() / tq["mid"] * 1e4
ax = tq.set_index("time")["eff_spread_bps"].resample("5min").mean().plot(title="Average effective spread (bps)")
ax.set_xlabel(""); plt.show()

## 3. Polars: the same job, faster

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
import polars as pl
pt = pl.from_pandas(trades)
vwap_expr = (pl.col("price") * pl.col("size")).sum() / pl.col("size").sum()
vwap_pl = None
if vwap_expr is not Ellipsis:
    out = pt.sort("time").group_by_dynamic("time", every="5m").agg(vwap_expr.alias("vwap")).to_pandas()
    vwap_pl = out.set_index("time")["vwap"]
ref = trades.set_index("time").assign(pv=lambda d: d["price"] * d["size"]).resample("5min")[["pv", "size"]].sum()
ref = (ref["pv"] / ref["size"]).dropna().rename("vwap")
vwap = p.check("VWAP with Polars", vwap_pl, ref)

In [ ]:
rng = np.random.default_rng(1)
n = 3_000_000
big = pd.DataFrame({"symbol": rng.choice([f"S{i}" for i in range(500)], n), "price": rng.uniform(10, 500, n),
                    "size": rng.integers(1, 1000, n)})
bpl = pl.from_pandas(big)
t_pd = p.timeit(lambda: big.assign(pv=big["price"] * big["size"]).groupby("symbol")[["pv", "size"]].sum(), repeat=3)
t_pl = p.timeit(lambda: bpl.group_by("symbol").agg((pl.col("price") * pl.col("size")).sum(), pl.col("size").sum()), repeat=3)
print(f"VWAP per symbol over {n:,} rows: pandas {t_pd * 1e3:.0f} ms, Polars {t_pl * 1e3:.0f} ms")

## Questions
1. Why is `direction="nearest"` wrong for a backtest even though it looks "more accurate"?
2. A resample on naive local timestamps misbehaves twice a year. Why do the timestamps here carry UTC?
3. When would you keep pandas rather than switch to Polars?

**Graded version:** `labs/part03/week11_data` (resampling, `merge_asof`, Polars lazy SMA).